# wheat-infer-yolox-s-6e-768-conf030

COCO事前学習済みYOLOX-S、768px、6 epoch、横反転0.5、seed 42。単体推論、confidence 0.30、NMS IoU 0.50。固定検証636枚のWheat Image Precisionは0.345790。比較対象はMobileNetV3 Faster R-CNN単体の提出58331537と、D2＋Faster R-CNNの提出58342094。

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

spec = json.loads('{"notebook": "yutarotaro/wheat-infer-yolox-s-6e-768-conf030", "source_notebook": "submission-yolox-s-6e-768-conf030.ipynb", "run_label": "yolox-s-6e-768-conf030", "description": "COCO事前学習済みYOLOX-S、768px、6 epoch、横反転0.5、seed 42。単体推論、confidence 0.30、NMS IoU 0.50。固定検証636枚のWheat Image Precisionは0.345790。比較対象はMobileNetV3 Faster R-CNN単体の提出58331537と、D2＋Faster R-CNNの提出58342094。", "wheel_dataset": "yutarotaro/wheat-yolox-offline-runtime", "wheels": {"loguru-0.7.3-py3-none-any.whl": "31a33c10c8e1e10422bfd431aeb5d351c7cf7fa671e3c4df004162264b28220c", "thop-0.1.1.post2209072238-py3-none-any.whl": "01473c225231927d2ad718351f78ebf7cffe6af3bed464c4f1ba1ef0f7cdda27"}, "yolox_source": {"file": "wheat-yolox-source.bin", "sha256": "68e5d3abea0337c92e051816edd9182d78aa67424ac3cfea32664ea8f32e7e81"}, "models": [{"trial_id": "602ac2d18a3042dc-0001", "notebook": "yutarotaro/wheat-train-yolox-s-6e-768-hflip-3c34d8001e01/1", "sha256": "a268feb6df5711421084de9f4abc5dbbf641355cb4049ce89e18ccb29bf8faf5", "config": {"detector": "yolox-s", "image_size": 768, "run_label": "yolox-s-6e-768-hflip", "epochs": 6, "batch_size": 4, "learning_rate": 0.0001, "horizontal_flip": 0.5, "vertical_flip": 0, "color_jitter": 0, "mosaic": 0, "validation_loss": true, "confidence": 0.3, "nms": 0.5}}], "confidence": 0.3, "validation_score": 0.34578952575630983}')
Path("submission-spec.json").write_text(json.dumps(spec))


def checked_input(name, expected):
    (path,) = Path("/kaggle/input").rglob(name)
    with path.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != expected:
            raise ValueError(f"Mounted file checksum mismatch: {name}")
    return path


wheel_paths = [str(checked_input(name, expected)) for name, expected in spec["wheels"].items()]
if wheel_paths:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                    "--force-reinstall", *wheel_paths], check=True)
if "yolox_source" in spec:
    source = spec["yolox_source"]
    cache = Path(".cache/yolox")
    cache.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(checked_input(source["file"], source["sha256"]), cache / "source.zip")
    os.environ["ML_LOOP_PROJECT"] = str(Path.cwd())


## data.py

In [ ]:
%%writefile data.py
"""Validate annotations and freeze a split by acquisition source."""

import argparse
import csv
import hashlib
import itertools
import json
import math
import random
from collections import Counter
from pathlib import Path

from PIL import Image


def sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def load_dataset(root):
    images = {}
    with (root / "train.csv").open(newline="") as handle:
        for row in csv.DictReader(handle):
            image_id, source = row["image_id"], row["source"]
            if not image_id or not source or Path(image_id).name != image_id:
                raise ValueError(f"Invalid image id or source: {image_id!r}, {source!r}")
            width, height = int(row["width"]), int(row["height"])
            if min(width, height) <= 0:
                raise ValueError(f"Invalid image dimensions: {image_id}")
            x, y, w, h = map(float, json.loads(row["bbox"]))
            if not all(map(math.isfinite, [x, y, w, h])) or min(x, y) < 0 or min(w, h) <= 0:
                raise ValueError(f"Invalid bounding box: {image_id}")
            if x + w > width + 1e-6 or y + h > height + 1e-6:
                raise ValueError(f"Out-of-bounds annotation: {image_id}")
            record = images.setdefault(image_id, dict(source=source, width=width, height=height, boxes=[]))
            if (record["source"], record["width"], record["height"]) != (source, width, height):
                raise ValueError(f"Inconsistent metadata: {image_id}")
            record["boxes"].append([x, y, min(x + w, width), min(y + h, height)])

    paths = {p.stem: p for p in (root / "train").glob("*.jpg")}
    missing = images.keys() - paths.keys()
    if missing:
        raise ValueError(f"Missing annotated images: {sorted(missing)[:5]}")
    unlabelled = sorted(paths.keys() - images.keys())
    if unlabelled:
        print(f"Excluding {len(unlabelled)} images with no source/annotation in train.csv.")
    checksums = {}
    for image_id, record in sorted(images.items()):
        path = paths[image_id]
        with Image.open(path) as im:
            if im.size != (record["width"], record["height"]):
                raise ValueError(f"Unexpected image dimensions: {image_id}")
        checksums[image_id] = sha(path)
    return images, checksums


def split_dataset(images, checksums, seed=42):
    parent = {r["source"]: r["source"] for r in images.values()}

    def group(s):
        while s != parent[s]:
            s = parent[s]
        return s

    seen = {}
    for image_id in sorted(images):
        source = images[image_id]["source"]
        previous = seen.setdefault(checksums[image_id], source)
        parent[group(source)] = group(previous)

    sizes = Counter(group(row["source"]) for row in images.values())
    groups = sorted(sizes)
    if len(groups) < 3:
        raise ValueError("Need at least three independent source groups.")
    rng = random.Random(seed)
    rng.shuffle(groups)
    target = (0.6, 0.2, 0.2)
    if len(groups) <= 10:
        assignments = itertools.product(range(3), repeat=len(groups))
    else:
        assignments = (rng.choices(range(3), target, k=len(groups)) for _ in range(10_000))
    winner, error = None, math.inf
    for assignment in assignments:
        if set(assignment) != {0, 1, 2}:
            continue
        counts = [sum(sizes[g] for g, part in zip(groups, assignment) if part == i) for i in range(3)]
        if counts[0] < max(counts[1:]):
            continue
        loss = sum((n / len(images) - goal) ** 2 for n, goal in zip(counts, target))
        if loss < error:
            winner, error = assignment, loss
    if winner is None:
        raise ValueError("No valid source-group split.")
    labels = {
        g: ("train", "validation", "holdout")[part] for g, part in zip(groups, winner)}
    partitions = {part: [] for part in ("train", "validation", "holdout")}
    for image_id, row in sorted(images.items()):
        partitions[labels[group(row["source"])]].append(image_id)
    return partitions


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, default=Path("data/raw"))
    parser.add_argument("--output", type=Path, default=Path("data/prepared/split.json"))
    args = parser.parse_args()
    if args.output.exists():
        parser.error("The split is already frozen; use a different output path to create a new evaluation cohort.")
    images, hashes = load_dataset(args.data)
    split = split_dataset(images, hashes)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    payload = {"seed": 42, "csv_sha256": sha(args.data / "train.csv"), "images": hashes, "partitions": split}
    with args.output.open("x") as handle:
        json.dump(payload, handle, sort_keys=True, indent=2)
    for part, ids in split.items():
        print(part, len(ids), dict(Counter(images[i]["source"] for i in ids)))


if __name__ == "__main__":
    main()


## metric.py

In [ ]:
%%writefile metric.py
"""Image-averaged wheat precision over IoU thresholds 0.50–0.75."""

import math

THRESHOLDS = tuple(n / 100 for n in range(50, 76, 5))


def area(box):
    return (box[2] - box[0]) * (box[3] - box[1])


def iou(a, b):
    overlap = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return overlap / (area(a) + area(b) - overlap)


def validate_predictions(rows, width, height):
    if not isinstance(rows, list):
        raise ValueError("Predictions must be a list.")
    for row in rows:
        if not isinstance(row, list) or len(row) != 5 or not all(
            isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) for x in row
        ):
            raise ValueError(f"Expected finite [confidence,x1,y1,x2,y2]: {row!r}")
        confidence, x1, y1, x2, y2 = row
        if not (0 <= confidence <= 1 and 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError(f"Invalid prediction: {row!r}")


def match(actual, predicted, threshold):
    remaining = set(range(len(actual)))
    hits, extras, duplicates = [], [], []
    for j in sorted(range(len(predicted)), key=lambda n: -predicted[n][0]):
        box = predicted[j][1:]
        overlaps = [iou(box, truth) for truth in actual]
        i = max(remaining, key=lambda k: (overlaps[k], -k), default=None)
        if i is not None and overlaps[i] > threshold:
            remaining.remove(i)
            hits.append(j)
        elif max(overlaps, default=0) > threshold:
            duplicates.append(j)
        else:
            extras.append(j)
    return hits, extras, duplicates, sorted(remaining)


def image_score(actual, predicted):
    if not actual and not predicted:
        return None
    scores = []
    for threshold in THRESHOLDS:
        hits, extra, duplicates, misses = match(actual, predicted, threshold)
        scores.append(len(hits) / (len(hits) + len(extra) + len(duplicates) + len(misses)))
    return sum(scores) / len(scores)


## fusion.py

In [ ]:
%%writefile fusion.py
"""Weighted Boxes Fusion for validated wheat detections in image coordinates."""

from ensemble_boxes.ensemble_boxes_wbf import weighted_boxes_fusion

from metric import validate_predictions


def fuse(rows, width, height, weights, threshold):
    scale = (width, height, width, height)
    for guesses in rows:
        validate_predictions(guesses, width, height)
    # Zero-confidence boxes carry no weight when averaging positions.
    rows = [[box for box in guesses if box[0] > 0] for guesses in rows]
    boxes, scores, _ = weighted_boxes_fusion(
        [[[value / size for value, size in zip(box[1:], scale)] for box in guesses] for guesses in rows],
        [[box[0] for box in guesses] for guesses in rows],
        [[1] * len(guesses) for guesses in rows],
        weights=weights, iou_thr=threshold, conf_type="box_and_model_avg",
    )
    # The library uses float32 cluster averages, which can round above 1.
    return [[min(float(conf), 1.0), *(min(float(value), 1.0) * size for value, size in zip(box, scale))]
            for box, conf in zip(boxes, scores)]


## model.py

In [ ]:
%%writefile model.py
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

import torch
from torch import nn
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms.functional import normalize, resize

EFFDET_PACKAGES = {"effdet": "0.4.1", "timm": "1.0.22"}


def prepare_efficientdet():
    missing = []
    for name, expected in EFFDET_PACKAGES.items():
        try:
            installed = version(name)
        except PackageNotFoundError:
            installed = None
        if installed != expected:
            missing.append(f"{name}=={expected}")
    if missing:
        if os.environ.get("ML_LOOP_REMOTE") != "kaggle":
            raise RuntimeError("Install the wheat group with uv sync --group wheat.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        *missing], check=True)


class EfficientWheat(nn.Module):
    def __init__(self, config, pretrained):
        super().__init__()
        prepare_efficientdet()
        from effdet import DetBenchPredict, DetBenchTrain, create_model

        self.size = config["image_size"]
        self.confidence = config["confidence"]
        if config["nms"] != 0.5:
            raise ValueError("EfficientDet uses NMS at IoU 0.50.")
        if self.size % 128:
            raise ValueError("EfficientDet image size must be a multiple of 128.")
        network = create_model("tf_efficientdet_d2", num_classes=1, pretrained=pretrained,
                               pretrained_backbone=False, image_size=(self.size, self.size),
                               max_det_per_image=300)
        self.loss_bench = DetBenchTrain(network)
        self.predict_bench = DetBenchPredict(network)

    def forward(self, images, labels=None):
        shapes = [image.shape[-2:] for image in images]
        inputs = torch.stack([
            normalize(resize(image, [self.size, self.size], antialias=True),
                      [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            for image in images
        ])
        if labels is not None:
            boxes = []
            for label, (h, w) in zip(labels, shapes, strict=True):
                gt = label["boxes"][:, [1, 0, 3, 2]]
                boxes.append(gt * gt.new_tensor([self.size / h, self.size / w] * 2))
            info = {
                "bbox": boxes, "cls": [label["labels"] for label in labels],
                "img_scale": inputs.new_ones(len(images)),
                "img_size": inputs.new_full((len(images), 2), self.size),
            }
            return {"loss": self.loss_bench(inputs, info)["loss"]}

        detections = self.predict_bench(inputs)
        outputs = []
        for prediction, (h, w) in zip(detections, shapes, strict=True):
            if not torch.isfinite(prediction).all():
                raise ValueError("Non-finite EfficientDet prediction.")
            scores = prediction[:, 4]
            boxes = prediction[:, :4].clamp(0, self.size)
            boxes = boxes * boxes.new_tensor([w / self.size, h / self.size] * 2)
            keep = (scores >= self.confidence) & (boxes[:, 2:] > boxes[:, :2]).all(dim=1)
            outputs.append({"boxes": boxes[keep], "scores": scores[keep]})
        return outputs


def create_model(config, pretrained=True):
    detector = config.get("detector", "fasterrcnn-mobilenet-v3")
    if detector == "yolox-s":
        from yolox_model import YoloWheat

        return YoloWheat(config, pretrained)
    if detector == "efficientdet-d2":
        return EfficientWheat(config, pretrained)
    if detector != "fasterrcnn-mobilenet-v3":
        raise ValueError(f"Unknown detector: {detector}")
    weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1 if pretrained else None
    model = fasterrcnn_mobilenet_v3_large_fpn(
        weights=weights, weights_backbone=None, min_size=config["image_size"],
        max_size=config["image_size"], box_score_thresh=config["confidence"],
        box_nms_thresh=config["nms"], box_detections_per_img=300,
    )
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, 2)
    return model


## submission.py

In [ ]:
%%writefile submission.py
"""Write Kaggle predictions from frozen detector checkpoints."""

import csv
import json
import math
from pathlib import Path

import torch
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

from data import sha
from metric import validate_predictions
from model import create_model


def kaggle_inputs(mounted, spec):
    roots = sorted(path.parent for path in mounted.rglob("sample_submission.csv")
                   if path.parent.name == "global-wheat-detection")
    if len(roots) != 1:
        raise ValueError("Expected one mounted competition dataset.")
    by_hash = {sha(path): path for path in mounted.rglob("weights.pt")}
    return roots[0], [by_hash[model["sha256"]] for model in spec["models"]]


def prediction_string(predictions, width, height):
    validate_predictions(predictions, width, height)
    tokens = []
    for score, x1, y1, x2, y2 in predictions:
        left, top = math.floor(x1), math.floor(y1)
        right, bottom = math.ceil(x2), math.ceil(y2)
        tokens.append(f"{score:.8f} {left} {top} {right-left} {bottom-top}")
    return " ".join(tokens)


@torch.inference_mode()
def predict(data_root, checkpoints, spec, output, device):
    if len(checkpoints) != len(spec["models"]):
        raise ValueError("Expected one checkpoint per model.")
    if len(checkpoints) > 1:
        from fusion import fuse

    models = []
    for path, conditions in zip(checkpoints, spec["models"], strict=True):
        if sha(path) != conditions["sha256"]:
            raise ValueError(f"Checkpoint mismatch for {conditions['trial_id']}.")
        model = create_model(conditions["config"], pretrained=False).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        models.append(model.eval())
    paths = {path.stem: path for path in (data_root / "test").glob("*.jpg")}
    with (data_root / "sample_submission.csv").open(newline="") as handle:
        ids = [row["image_id"] for row in csv.DictReader(handle)]
    if not ids or len(ids) != len(paths) or set(ids) != paths.keys():
        raise ValueError("Sample submission and test images must contain the same unique IDs.")
    temporary = output.with_suffix(".tmp")
    with temporary.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(("image_id", "PredictionString"))
        for key in ids:
            with Image.open(paths[key]) as image:
                width, height = image.size
                tensor = pil_to_tensor(image.convert("RGB")).float().div_(255).to(device)
            results = [model([tensor])[0] for model in models]
            rows = [[[score, *box] for score, box in zip(result["scores"].tolist(), result["boxes"].tolist(),
                                                          strict=True)] for result in results]
            if len(rows) == 1:
                detections = rows[0]
            else:
                detections = fuse(rows, width, height, spec["weights"], spec["iou"])
            detections = [box for box in detections if box[0] >= spec["confidence"]]
            writer.writerow((key, prediction_string(detections, width, height)))
    temporary.replace(output)
    print(f"Validated and wrote {len(ids)} predictions")


def run_kaggle(spec_path):
    spec = json.loads(spec_path.read_text())
    data, checkpoints = kaggle_inputs(Path("/kaggle/input"), spec)
    torch.set_num_threads(2)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    predict(data, checkpoints, spec, Path("/kaggle/working/submission.csv"), device)


## yolox_model.py

In [ ]:
%%writefile yolox_model.py
"""YOLOX-S using the same wheat training and prediction contract as the other detectors."""

import hashlib
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import torch
from torch import nn
from torchvision.ops import nms
from torchvision.transforms.functional import resize

SOURCE_REVISION = "419778480ab6ec0590e5d3831b3afb3b46ab2aa3"
SOURCE_URL = f"https://codeload.github.com/Megvii-BaseDetection/YOLOX/zip/{SOURCE_REVISION}"
SOURCE_SHA256 = "68e5d3abea0337c92e051816edd9182d78aa67424ac3cfea32664ea8f32e7e81"
WEIGHTS_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth"
WEIGHTS_SHA256 = "f55ded7181e1b0c13285c56e7790b8f0e8f8db590fe4edb37f0b7f345c913a30"


def fetch(path, url, expected):
    if not path.is_file():
        temporary = path.with_suffix(".download")
        urlretrieve(url, temporary)
        temporary.replace(path)
    with path.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != expected:
            raise ValueError(f"Unexpected SHA-256 for {path.name}.")
    return path


def prepare_yolox():
    missing = [package for module, package in (("cv2", "opencv-python-headless==4.13.0.92"),
                                                ("loguru", "loguru==0.7.3"),
                                                ("thop", "thop==0.1.1.post2209072238"))
               if find_spec(module) is None]
    if missing:
        if os.environ.get("ML_LOOP_REMOTE") != "kaggle":
            raise RuntimeError("Install YOLOX dependencies with uv sync --group wheat --group yolo.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        *missing], check=True)
    project = Path(os.environ.get("ML_LOOP_PROJECT", Path(__file__).resolve().parents[2]))
    cache = project / ".cache/yolox"
    cache.mkdir(parents=True, exist_ok=True)
    source = fetch(cache / "source.zip", SOURCE_URL, SOURCE_SHA256)
    source_root = f"YOLOX-{SOURCE_REVISION}"
    with ZipFile(source) as archive:
        archive.extractall(cache, (name for name in archive.namelist()
                                   if name.startswith(f"{source_root}/yolox/")))
    sys.path.insert(0, str(cache / source_root))
    return cache


class YoloWheat(nn.Module):
    def __init__(self, config, pretrained):
        super().__init__()
        cache = prepare_yolox()
        from yolox.models import YOLOPAFPN, YOLOXHead

        self.size = config["image_size"]
        self.confidence = config["confidence"]
        self.nms = config["nms"]
        if self.size % 32:
            raise ValueError("YOLOX image size must be a multiple of 32.")
        self.backbone = YOLOPAFPN(depth=0.33, width=0.5)
        self.loss_bench = YOLOXHead(num_classes=1, width=0.5)
        for module in self.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eps = 1e-3
                module.momentum = 0.03
        self.loss_bench.initialize_biases(1e-2)
        if pretrained:
            path = fetch(cache / "yolox_s.pth", WEIGHTS_URL, WEIGHTS_SHA256)
            state = torch.load(path, map_location="cpu", weights_only=True)["model"]
            backbone = {key.removeprefix("backbone."): value for key, value in state.items()
                        if key.startswith("backbone.")}
            head = {key.removeprefix("head."): value for key, value in state.items()
                    if key.startswith("head.") and not key.startswith("head.cls_preds.")}
            self.backbone.load_state_dict(backbone)
            result = self.loss_bench.load_state_dict(head, strict=False)
            expected = {key for key in self.loss_bench.state_dict() if key.startswith("cls_preds.")}
            if set(result.missing_keys) != expected or result.unexpected_keys:
                raise ValueError(f"YOLOX pretrained head does not match: {result}.")

    def forward(self, images, labels=None):
        shapes = [image.shape[-2:] for image in images]
        # The published YOLOX-S weights use BGR inputs on the 0–255 scale.
        inputs = torch.stack([resize(image.flip(0), [self.size, self.size], antialias=True)
                              for image in images]) * 255
        features = self.backbone(inputs)
        if labels is not None:
            count = max(1, max(len(target["boxes"]) for target in labels))
            targets = inputs.new_zeros((len(labels), count, 5))
            for i, (target, (h, w)) in enumerate(zip(labels, shapes, strict=True)):
                boxes = target["boxes"] * inputs.new_tensor([self.size / w, self.size / h] * 2)
                centers = (boxes[:, :2] + boxes[:, 2:]) / 2
                targets[i, :len(boxes), 1:3] = centers
                targets[i, :len(boxes), 3:] = boxes[:, 2:] - boxes[:, :2]
            loss, *_ = self.loss_bench(features, targets, inputs)
            return {"loss": loss}

        outputs = []
        for prediction, (h, w) in zip(self.loss_bench(features), shapes, strict=True):
            if not torch.isfinite(prediction).all():
                raise ValueError("Non-finite YOLOX prediction.")
            scores = prediction[:, 4] * prediction[:, 5]
            qualified = scores >= self.confidence
            scores = scores[qualified]
            centers = prediction[qualified, :2]
            half_sizes = prediction[qualified, 2:4] / 2
            boxes = torch.cat((centers - half_sizes, centers + half_sizes), dim=1)
            boxes = boxes.clamp(0, self.size) * boxes.new_tensor([w / self.size, h / self.size] * 2)
            valid = (boxes[:, 2:] > boxes[:, :2]).all(dim=1)
            boxes, scores = boxes[valid], scores[valid]
            keep = nms(boxes, scores, self.nms)[:300]
            outputs.append({"boxes": boxes[keep], "scores": scores[keep]})
        return outputs


## 推論と提出用CSVの作成

In [ ]:
from submission import run_kaggle

run_kaggle(Path("submission-spec.json"))
